# 05 - Probability Calibration and Threshold Analysis

This notebook analyzes the probability outputs of the current main conflict risk model.

The goal is to evaluate whether the default classification threshold of `0.5` is adequate and how different thresholds affect precision, recall and F1-score.

This step is important because the project aims to evolve from a binary classifier into an experimental risk analysis system.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


current_path = Path.cwd()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parents[1]

PREDICTIONS_PATH = PROJECT_ROOT / "outputs" / "tables" / "conflict_risk_model_test_predictions.csv"

predictions = pd.read_csv(PREDICTIONS_PATH)

print("Predictions loaded successfully.")
print("Shape:", predictions.shape)
print("Columns:")
print(list(predictions.columns))

predictions.head()

Predictions loaded successfully.
Shape: (1358, 9)
Columns:
['country', 'year', 'region', 'world_bank_country_code', 'world_bank_country_name', 'target_conflict_next_year', 'organized_violence_exists', 'predicted_conflict_next_year', 'predicted_conflict_probability']


,country,year,region,world_bank_country_code,world_bank_country_name,target_conflict_next_year,organized_violence_exists,predicted_conflict_next_year,predicted_conflict_probability
0,Afghanistan,2017,Asia,AFG,Afghanistan,1,1,1,0.999989
1,Afghanistan,2018,Asia,AFG,Afghanistan,1,1,1,0.999568
2,Afghanistan,2019,Asia,AFG,Afghanistan,1,1,1,0.999454
3,Afghanistan,2020,Asia,AFG,Afghanistan,1,1,1,0.999123
4,Afghanistan,2021,Asia,AFG,Afghanistan,1,1,1,0.999723


In [2]:
TARGET_COLUMN = "target_conflict_next_year"
PROBABILITY_COLUMN = "predicted_conflict_probability"

y_true = predictions[TARGET_COLUMN]
y_proba = predictions[PROBABILITY_COLUMN]

print("Probability summary:")
print(y_proba.describe().round(4))

print("\nTarget distribution:")
print(y_true.value_counts(normalize=True).sort_index().round(4))

print("\nGlobal probability metrics:")
print("ROC-AUC:", round(roc_auc_score(y_true, y_proba), 4))
print("Average precision:", round(average_precision_score(y_true, y_proba), 4))
print("Brier score:", round(brier_score_loss(y_true, y_proba), 4))

Probability summary:
count    1358.0000
mean        0.3435
std         0.3973
min         0.0001
25%         0.0311
50%         0.1012
75%         0.8356
max         1.0000
Name: predicted_conflict_probability, dtype: float64

Target distribution:
target_conflict_next_year
0    0.6753
1    0.3247
Name: proportion, dtype: float64

Global probability metrics:
ROC-AUC: 0.9647
Average precision: 0.9472
Brier score: 0.0626


In [3]:
def evaluate_threshold(threshold):
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)

threshold_results = pd.DataFrame(
    [evaluate_threshold(threshold) for threshold in thresholds]
)

threshold_results.sort_values(
    by=["f1_score", "recall", "precision"],
    ascending=False,
).round(4)

,threshold,accuracy,precision,recall,f1_score,tn,fp,fn,tp
8,0.50,0.9175,0.8926,0.8481,0.8698,872,45,67,374
7,0.45,0.9161,0.8811,0.8571,0.8690,866,51,63,378
9,0.55,0.9161,0.8978,0.8367,0.8662,875,42,72,369
4,0.30,0.9094,0.8354,0.8980,0.8656,839,78,45,396
10,0.60,0.9168,0.9121,0.8231,0.8653,882,35,78,363
6,0.40,0.9116,0.8656,0.8617,0.8636,858,59,61,380
5,0.35,0.9102,0.8537,0.8730,0.8632,851,66,56,385
11,0.65,0.9146,0.9177,0.8095,0.8602,885,32,84,357
3,0.25,0.9021,0.8117,0.9093,0.8578,824,93,40,401
12,0.70,0.9131,0.9261,0.7959,0.8561,889,28,90,351


In [4]:
default_threshold_result = threshold_results[
    threshold_results["threshold"] == 0.50
].copy()

best_f1_result = threshold_results.sort_values(
    by=["f1_score", "recall", "precision"],
    ascending=False,
).head(1)

print("Default threshold:")
print(default_threshold_result.round(4).to_string(index=False))

print("\nBest F1 threshold:")
print(best_f1_result.round(4).to_string(index=False))

Default threshold:
 threshold  accuracy  precision  recall  f1_score  tn  fp  fn  tp
       0.5    0.9175     0.8926  0.8481    0.8698 872  45  67 374

Best F1 threshold:
 threshold  accuracy  precision  recall  f1_score  tn  fp  fn  tp
       0.5    0.9175     0.8926  0.8481    0.8698 872  45  67 374


In [5]:
bins = np.linspace(0, 1, 11)

calibration_table = predictions.copy()
calibration_table["probability_bin"] = pd.cut(
    calibration_table[PROBABILITY_COLUMN],
    bins=bins,
    include_lowest=True,
)

calibration_summary = (
    calibration_table
    .groupby("probability_bin", observed=False)
    .agg(
        samples=(TARGET_COLUMN, "size"),
        mean_predicted_probability=(PROBABILITY_COLUMN, "mean"),
        observed_positive_rate=(TARGET_COLUMN, "mean"),
    )
    .reset_index()
)

calibration_summary["calibration_error"] = (
    calibration_summary["mean_predicted_probability"]
    - calibration_summary["observed_positive_rate"]
)

calibration_summary.round(4)

,probability_bin,samples,mean_predicted_probability,observed_positive_rate,calibration_error
0,"(-0.001, 0.1]",677,0.0403,0.0222,0.0182
1,"(0.1, 0.2]",149,0.1375,0.1007,0.0368
2,"(0.2, 0.3]",58,0.2401,0.2586,-0.0185
3,"(0.3, 0.4]",35,0.3389,0.4571,-0.1182
4,"(0.4, 0.5]",20,0.4486,0.3000,0.1486
5,"(0.5, 0.6]",21,0.5476,0.5238,0.0238
6,"(0.6, 0.7]",19,0.6524,0.6316,0.0209
7,"(0.7, 0.8]",28,0.7584,0.5714,0.1870
8,"(0.8, 0.9]",50,0.8632,0.8400,0.0232
9,"(0.9, 1.0]",301,0.9823,0.9734,0.0089


In [6]:
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

threshold_output_path = OUTPUT_TABLES_DIR / "probability_threshold_results.csv"
calibration_output_path = OUTPUT_TABLES_DIR / "probability_calibration_bins.csv"

threshold_results.to_csv(threshold_output_path, index=False)
calibration_summary.to_csv(calibration_output_path, index=False)

print(f"Saved threshold results to: {threshold_output_path}")
print(f"Saved calibration bins to: {calibration_output_path}")

Saved threshold results to: C:\Users\enzo.going\Documents\GitHub\international-conflict-risk-ml\outputs\tables\probability_threshold_results.csv
Saved calibration bins to: C:\Users\enzo.going\Documents\GitHub\international-conflict-risk-ml\outputs\tables\probability_calibration_bins.csv


## Interpretation notes

The threshold analysis should answer:

- Is `0.5` the best threshold for F1-score?
- What happens to precision and recall when the threshold is reduced?
- What happens when the threshold is increased?
- Are the predicted probabilities reasonably aligned with observed frequencies?

This is not yet full probability calibration with a separate calibration model. It is the first diagnostic step before deciding whether calibration methods such as Platt scaling or isotonic regression are needed.